# Notebook 03 — Streaming e Gestione Robusta degli Errori

**Settimana:** 3
**Blocchi:** 3.3 Streaming · 3.4 Errori, rate limit, retry
**Tempo stimato:** 2.5-3 ore

## Cosa costruirai

- Streaming in tempo reale su OpenAI e Anthropic
- Accumulo del testo completo con metriche di latenza
- Un client LLM robusto con retry automatici usando `tenacity`
- Simulazione di errori senza consumare token
- Una classe `LLMClient` pronta per la produzione che incapsula tutto

## Prerequisiti

```bash
pip install openai anthropic python-dotenv tenacity
```

---
## Cella 0 — Setup

In [ ]:
import os
import time
import random
import logging
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError, BadRequestError
import anthropic
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    wait_random,
    retry_if_exception_type,
    before_sleep_log,
    RetryError
)

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=30.0)
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"), timeout=30.0)

print("Setup completato.")

---
## Cella 1 — Streaming di base con OpenAI

Esegui questa cella e osserva i token apparire in tempo reale.
Misura anche la latenza fino al primo token (TTFT) e il tempo totale.

In [ ]:
PREGUNTA = "Spiega in 4 paragrafi come funziona l'attenzione nei Transformer. Sii dettagliato."

print(f"Domanda: {PREGUNTA}")
print("\n" + "="*60 + "\n")

t_inicio         = time.perf_counter()
t_primer_token   = None
texto_completo   = ""
n_chunks         = 0

stream = cliente_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PREGUNTA}],
    stream=True,
    stream_options={"include_usage": True}   # richiede i token nell'ultimo chunk
)

uso = None
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None

    if delta:
        if t_primer_token is None:
            t_primer_token = time.perf_counter()    # primo token ricevuto
        texto_completo += delta
        n_chunks       += 1
        print(delta, end="", flush=True)

    if chunk.usage:       # ultimo chunk
        uso = chunk.usage

t_fin = time.perf_counter()

# ── Metriche ──────────────────────────────────────────────────────────────────
ttft          = (t_primer_token - t_inicio) * 1000 if t_primer_token else 0
tiempo_total  = (t_fin - t_inicio) * 1000
tokens_output = uso.completion_tokens if uso else "N/D"
tps           = uso.completion_tokens / (t_fin - t_inicio) if uso else 0

print(f"\n\n{'='*60}")
print(f"Tempo fino al primo token (TTFT): {ttft:.0f} ms")
print(f"Tempo totale di generazione:      {tiempo_total:.0f} ms")
print(f"Chunk ricevuti:                   {n_chunks}")
print(f"Token output:                     {tokens_output}")
print(f"Velocità:                         {tps:.1f} token/s")
print(f"Lunghezza del testo:               {len(texto_completo)} caratteri")

**Osservazioni** (modifica questa cella):

- TTFT ottenuto: ... ms — è quello che ti aspettavi?
- Quanti chunk sono arrivati? Coincide con il numero di token?
- Confronta questo tempo con una chiamata non-stream (eseguila tu e misura): ...

---
## Cella 2 — Streaming con Anthropic

L'SDK di Anthropic usa un context manager (`with ... as stream`).
`stream.text_stream` è l'iteratore di alto livello che ti dà solo il testo.
Uscendo dal `with`, `stream.get_final_message()` contiene il messaggio completo con usage.

In [ ]:
print(f"Domanda: {PREGUNTA}")
print("\n" + "="*60 + "\n")

t_inicio       = time.perf_counter()
t_primer_token = None
texto_completo = ""

with cliente_anthropic.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[{"role": "user", "content": PREGUNTA}]
) as stream:
    for texto in stream.text_stream:
        if t_primer_token is None:
            t_primer_token = time.perf_counter()
        texto_completo += texto
        print(texto, end="", flush=True)

    mensaje_final = stream.get_final_message()

t_fin = time.perf_counter()

ttft         = (t_primer_token - t_inicio) * 1000 if t_primer_token else 0
tiempo_total = (t_fin - t_inicio) * 1000
tps          = mensaje_final.usage.output_tokens / (t_fin - t_inicio)

print(f"\n\n{'='*60}")
print(f"Tempo fino al primo token (TTFT): {ttft:.0f} ms")
print(f"Tempo totale:                     {tiempo_total:.0f} ms")
print(f"Token input / output:             {mensaje_final.usage.input_tokens} / {mensaje_final.usage.output_tokens}")
print(f"Velocità:                         {tps:.1f} token/s")
print(f"Stop reason:                      {mensaje_final.stop_reason}")

---
## Cella 3 — Simulare e gestire errori senza consumare token

Testare la gestione degli errori contro l'API reale è difficile (dovresti davvero superare il tuo rate limit).
La soluzione: crea funzioni che lancino le stesse eccezioni che lancerebbe l'API reale,
ma dal tuo stesso codice.

Così puoi testare la tua logica di retry senza chiamate reali.

In [ ]:
from openai import InternalServerError
from unittest.mock import patch, MagicMock
import httpx

# ── Fabbrica di errori finti ────────────────────────────────────────────────
# I costruttori delle eccezioni di OpenAI richiedono una risposta HTTP.
# Creiamo una risposta mock minima.

def crear_error_openai(tipo_error, mensaje="Errore simulato"):
    """Crea un'istanza di eccezione di OpenAI con una risposta HTTP finta."""
    respuesta_falsa = httpx.Response(
        status_code=429 if "RateLimit" in tipo_error.__name__ else 500,
        request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions")
    )
    return tipo_error(mensaje, response=respuesta_falsa, body={})


# ── Funzione che fallisce N volte prima di avere successo ────────────────────────────
# Simula un servizio saturo che si riprende dopo alcuni tentativi.

class SimuladorFallos:
    """Simula un endpoint che fallisce `n_fallos` volte prima di rispondere correttamente."""

    def __init__(self, n_fallos: int, tipo_error=RateLimitError):
        self.n_fallos   = n_fallos
        self.tipo_error = tipo_error
        self.intentos   = 0

    def llamar(self, mensaje: str) -> str:
        self.intentos += 1
        print(f"  [Simulatore] Tentativo {self.intentos}...")

        if self.intentos <= self.n_fallos:
            error = crear_error_openai(self.tipo_error, f"Errore simulato #{self.intentos}")
            print(f"  [Simulatore] Lancio {self.tipo_error.__name__}")
            raise error

        # Successo
        return f"Risposta reale dopo {self.intentos} tentativi: '{mensaje}'"


# ── Test senza retry — l'errore si propaga direttamente ─────────────────
print("=== Senza retry ===")
sim = SimuladorFallos(n_fallos=2)
try:
    resultado = sim.llamar("Ciao")
    print(f"Risultato: {resultado}")
except RateLimitError as e:
    print(f"Errore non catturato: {type(e).__name__} — l'app si è rotta")

---
## Cella 4 — Backoff esponenziale con `tenacity`

Ora aggiungi retry automatici. Osserva quanto attende tra un tentativo e l'altro.

In [ ]:
# ── Decoratore di retry ───────────────────────────────────────────────────
# Modifica questi parametri e osserva come cambia il comportamento.

ERRORES_REINTENTABLES = (
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    anthropic.RateLimitError,
    anthropic.APIConnectionError,
    anthropic.InternalServerError,
    anthropic.OverloadedError,
)

estrategia_reintento = retry(
    retry=retry_if_exception_type(ERRORES_REINTENTABLES),
    wait=wait_exponential(multiplier=1, min=1, max=10) + wait_random(0, 0.5),
    stop=stop_after_attempt(4),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True    # se i tentativi si esauriscono, propaga l'eccezione originale
)


@estrategia_reintento
def llamar_con_reintento(sim: SimuladorFallos, mensaje: str) -> str:
    """Avvolge qualsiasi chiamata con la strategia di retry."""
    return sim.llamar(mensaje)


# ── Caso 1: fallisce 2 volte, poi ha successo ───────────────────────────────────────
print("=== Caso 1: 2 fallimenti → successo ===")
sim1 = SimuladorFallos(n_fallos=2)
t0   = time.perf_counter()
try:
    resultado = llamar_con_reintento(sim1, "Test")
    print(f"Risultato: {resultado}")
    print(f"Tempo totale con attese: {(time.perf_counter() - t0):.2f}s")
except Exception as e:
    print(f"Fallito definitivamente: {e}")

print()

# ── Caso 2: fallisce più volte di quante ne permetta tenacity ──────────────────────
print("=== Caso 2: 5 fallimenti → retry esauriti ===")
sim2 = SimuladorFallos(n_fallos=5)
t0   = time.perf_counter()
try:
    resultado = llamar_con_reintento(sim2, "Test")
    print(f"Risultato: {resultado}")
except RateLimitError as e:
    print(f"Fallito definitivamente dopo {sim2.intentos} tentativi")
    print(f"Tempo totale: {(time.perf_counter() - t0):.2f}s")

print()

# ── Caso 3: errore NON ritentabile (BadRequestError) ──────────────────────────
print("=== Caso 3: BadRequestError — non deve ritentare ===")
sim3 = SimuladorFallos(n_fallos=1, tipo_error=BadRequestError)
try:
    resultado = llamar_con_reintento(sim3, "Test")
except BadRequestError:
    print(f"BadRequestError catturato dopo {sim3.intentos} tentativo/i — corretto, non ha ritentato")
except Exception as e:
    print(f"Errore inatteso: {type(e).__name__}")

**Osservazioni** (modifica questa cella):

- Nel caso 1, quanto ha atteso tra il tentativo 1 e il 2? E tra il 2 e il 3?
- Nel caso 3, ha ritentato? Perché no?
- Se aumenti `min=1` a `min=5`, cosa cambia?

Prova a modificare i parametri di `wait_exponential` e riesegui.

---
## Cella 5 — Classe `LLMClient` pronta per la produzione

Combina tutto ciò che hai imparato nelle settimane 1, 2 e 3 in una classe coesa.
Questa sarà la base di tutti i tuoi progetti del corso.

In [ ]:
from dataclasses import dataclass, field
from typing import Literal, Iterator, Callable


@dataclass
class LLMRespuesta:
    """Risposta normalizzata, indipendente dal provider."""
    texto:          str
    tokens_input:   int
    tokens_output:  int
    modelo:         str
    proveedor:      str
    latencia_ms:    float
    ttft_ms:        float | None = None

    @property
    def coste_usd(self) -> float:
        """Stima del costo. Aggiorna i prezzi se cambiano."""
        precios = {
            "gpt-4o-mini":              {"input": 0.150, "output": 0.600},
            "gpt-4o":                   {"input": 2.500, "output": 10.00},
            "claude-haiku-4-5-20251001":{"input": 0.080, "output": 0.400},
            "claude-sonnet-4-6":        {"input": 3.000, "output": 15.00},
        }
        p = precios.get(self.modelo, {"input": 0, "output": 0})
        return (
            (self.tokens_input  / 1_000_000) * p["input"] +
            (self.tokens_output / 1_000_000) * p["output"]
        )


class LLMClient:
    """
    Client unificato per OpenAI e Anthropic.

    Caratteristiche:
    - Streaming con metriche di latenza (TTFT e totale)
    - Retry automatici con backoff esponenziale
    - Risposta normalizzata in LLMRespuesta
    - Logging integrato
    """

    MODELOS_DEFAULT = {
        "openai":    "gpt-4o-mini",
        "anthropic": "claude-haiku-4-5-20251001",
    }

    def __init__(
        self,
        proveedor: Literal["openai", "anthropic"] = "anthropic",
        modelo: str | None = None,
        timeout: float = 30.0,
        max_reintentos: int = 3,
    ):
        self.proveedor       = proveedor
        self.modelo          = modelo or self.MODELOS_DEFAULT[proveedor]
        self.max_reintentos  = max_reintentos
        self._logger         = logging.getLogger(self.__class__.__name__)

        if proveedor == "openai":
            self._cliente = OpenAI(
                api_key=os.getenv("OPENAI_API_KEY"),
                timeout=timeout
            )
        else:
            self._cliente = anthropic.Anthropic(
                api_key=os.getenv("ANTHROPIC_API_KEY"),
                timeout=timeout
            )

        # Costruisce il decoratore di retry con i parametri dell'istanza
        self._reintento = retry(
            retry=retry_if_exception_type(ERRORES_REINTENTABLES),
            wait=wait_exponential(multiplier=1, min=2, max=60) + wait_random(0, 1),
            stop=stop_after_attempt(self.max_reintentos),
            before_sleep=before_sleep_log(self._logger, logging.WARNING),
            reraise=True
        )

    # ── Chiamata standard (senza streaming) ────────────────────────────────

    def completar(
        self,
        mensajes: list[dict],
        system: str | None = None,
        temperature: float = 0.7,
        max_tokens: int = 1024,
    ) -> LLMRespuesta:
        """Chiamata standard con retry. Restituisce LLMRespuesta."""

        @self._reintento
        def _llamar():
            t0 = time.perf_counter()
            if self.proveedor == "openai":
                msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
                r = self._cliente.chat.completions.create(
                    model=self.modelo, messages=msgs,
                    temperature=temperature, max_tokens=max_tokens
                )
                return LLMRespuesta(
                    texto=r.choices[0].message.content,
                    tokens_input=r.usage.prompt_tokens,
                    tokens_output=r.usage.completion_tokens,
                    modelo=self.modelo, proveedor=self.proveedor,
                    latencia_ms=(time.perf_counter() - t0) * 1000
                )
            else:
                kwargs = dict(
                    model=self.modelo, messages=mensajes,
                    temperature=temperature, max_tokens=max_tokens
                )
                if system:
                    kwargs["system"] = system
                r = self._cliente.messages.create(**kwargs)
                return LLMRespuesta(
                    texto=r.content[0].text,
                    tokens_input=r.usage.input_tokens,
                    tokens_output=r.usage.output_tokens,
                    modelo=self.modelo, proveedor=self.proveedor,
                    latencia_ms=(time.perf_counter() - t0) * 1000
                )

        return _llamar()

    # ── Chiamata con streaming ────────────────────────────────────────────────

    def completar_stream(
        self,
        mensajes: list[dict],
        system: str | None = None,
        max_tokens: int = 1024,
        on_token: Callable[[str], None] | None = None,
    ) -> LLMRespuesta:
        """
        Chiamata con streaming.
        `on_token`: callback opzionale chiamato per ogni token (es. per stampare).
        Restituisce LLMRespuesta con il testo completo al termine.
        """
        t0             = time.perf_counter()
        t_primer_token = None
        texto_completo = ""

        if self.proveedor == "openai":
            msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
            stream = self._cliente.chat.completions.create(
                model=self.modelo, messages=msgs, stream=True,
                max_tokens=max_tokens,
                stream_options={"include_usage": True}
            )
            uso = None
            for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    if t_primer_token is None:
                        t_primer_token = time.perf_counter()
                    texto_completo += delta
                    if on_token:
                        on_token(delta)
                if chunk.usage:
                    uso = chunk.usage
            tokens_in  = uso.prompt_tokens     if uso else 0
            tokens_out = uso.completion_tokens if uso else 0

        else:  # anthropic
            kwargs = dict(model=self.modelo, messages=mensajes, max_tokens=max_tokens)
            if system:
                kwargs["system"] = system
            with self._cliente.messages.stream(**kwargs) as stream:
                for token in stream.text_stream:
                    if t_primer_token is None:
                        t_primer_token = time.perf_counter()
                    texto_completo += token
                    if on_token:
                        on_token(token)
                final = stream.get_final_message()
            tokens_in  = final.usage.input_tokens
            tokens_out = final.usage.output_tokens

        ttft = (t_primer_token - t0) * 1000 if t_primer_token else None
        return LLMRespuesta(
            texto=texto_completo,
            tokens_input=tokens_in,
            tokens_output=tokens_out,
            modelo=self.modelo,
            proveedor=self.proveedor,
            latencia_ms=(time.perf_counter() - t0) * 1000,
            ttft_ms=ttft
        )


# ── Test della classe ────────────────────────────────────────────────────────

PREGUNTA_CORTA = "Qual è la differenza tra un processo e un thread nei sistemi operativi?"

# Test 1: chiamata standard
print("=== Chiamata standard (Anthropic) ===")
llm = LLMClient(proveedor="anthropic")
resultado = llm.completar(
    mensajes=[{"role": "user", "content": PREGUNTA_CORTA}],
    system="Rispondi in massimo due frasi."
)
print(f"Testo:       {resultado.texto}")
print(f"Latenza:     {resultado.latencia_ms:.0f} ms")
print(f"Token:       {resultado.tokens_input} in / {resultado.tokens_output} out")
print(f"Costo:       ${resultado.coste_usd:.6f}")

print()

# Test 2: streaming con callback
print("=== Streaming (OpenAI) ===")
llm_oai = LLMClient(proveedor="openai")
resultado_stream = llm_oai.completar_stream(
    mensajes=[{"role": "user", "content": PREGUNTA_CORTA}],
    on_token=lambda t: print(t, end="", flush=True)
)
print(f"\n\nTTFT:        {resultado_stream.ttft_ms:.0f} ms")
print(f"Latenza:     {resultado_stream.latencia_ms:.0f} ms")
print(f"Costo:       ${resultado_stream.coste_usd:.6f}")

---
## Sfida finale — Chat da console con streaming, retry e Ctrl+C pulito

Costruisci una chat da console usando `LLMClient` che:
- Mostri i token in streaming in tempo reale
- Stampi latenza e costo accumulato dopo ogni turno
- Premendo Ctrl+C, si chiuda in modo pulito mostrando il costo totale della sessione
- Se c'è un errore irrecuperabile (i retry si esauriscono), lo mostri senza interrompere il loop

Lo scheletro gestisce già Ctrl+C. Il tuo compito è completare i `# TODO`.

In [ ]:
def chat_streaming(
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system: str = "Sei un assistente utile. Rispondi sempre in italiano."
) -> None:
    """
    Chat da console con streaming e metriche per turno.
    Scrivi 'esci' per terminare, oppure premi Ctrl+C.
    """
    llm             = LLMClient(proveedor=proveedor)
    historial       = []
    coste_total     = 0.0
    n_turno         = 0

    print(f"Chat avviata | {proveedor} / {llm.modelo}")
    print("Scrivi 'esci' per terminare.\n")

    try:
        while True:
            # Lettura dell'input
            try:
                entrada = input("Tu: ").strip()
            except EOFError:
                break
            if not entrada:
                continue
            if entrada.lower() in ("esci", "exit", "quit"):
                break

            historial.append({"role": "user", "content": entrada})
            n_turno += 1

            print(f"\nAssistente: ", end="", flush=True)

            try:
                # TODO: chiama llm.completar_stream() con la cronologia completa.
                # Usa on_token per stampare ogni token in tempo reale.
                # Salva il risultato in una variabile `resultado`.
                resultado = None  # sostituisci questo

                # TODO: aggiungi la risposta alla cronologia
                # historial.append(...)

                # TODO: accumula il costo e mostra le metriche del turno
                # coste_total += resultado.coste_usd
                # print(f"\n[turno {n_turno} | {resultado.latencia_ms:.0f}ms | ...")

            except RetryError:
                print("\n[Errore] Impossibile ottenere una risposta dopo vari tentativi.")
            except Exception as e:
                print(f"\n[Errore inatteso] {type(e).__name__}: {e}")

    except KeyboardInterrupt:
        pass  # Ctrl+C — cade nel blocco finally

    finally:
        print(f"\n\n{'='*50}")
        print(f"Sessione terminata | {n_turno} turni")
        print(f"Costo totale: ${coste_total:.6f} USD")
        print(f"{'='*50}")


# Decommenta per eseguire (meglio da un .py in terminale che in Jupyter)
# chat_streaming(proveedor="anthropic")

### Estensioni opzionali

1. **Limite di contesto scorrevole:** quando la cronologia supera 20 messaggi, elimina i più vecchi mantenendo sempre il primo messaggio di sistema e gli ultimi N.

2. **Timeout visibile:** se il primo token non arriva entro 3 secondi, mostra un indicatore di `...` lampeggiante durante l'attesa.

3. **Esporta la sessione:** alla chiusura, salva la cronologia completa in `sesion_YYYYMMDD_HHMMSS.json`.

In [ ]:
# Spazio per estensioni opzionali

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 03 — Settimana 3 del Piano di Formazione AI Engineering*